# Data Center Energy Stress Predictor
## A Multi-Source Geospatial Analytics and Visualization Platform

**Course:** B.Tech IT — Data Analytics & Visualization
**Regions:** Virginia, Ohio (2015–2017) · Texas, Georgia (2019–2021)
**Sources:** PJM Kaggle CSVs · EIA-930 API · NREL NSRDB API · IM3 Data Center Atlas

Pipeline: collect → clean → integrate → feature engineering → EDA → stress index → geospatial → dashboard.
No machine learning or deep learning. All derived metrics are labeled as such.

> **Note on the name:** despite "Predictor" in the title, this project performs **no prediction or
> forecasting**. All results are descriptive statistics of historical data.
>
> **Synthetic data:** Section 7 adds 20,000 clearly labelled *synthetic demonstration* rows (`data_type == "synthetic"`) for the dashboard's demo mode. Every analysis in this notebook uses real data only.

In [1]:
import os, json, zipfile, io, time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

load_dotenv()
EIA_KEY    = os.environ["EIA_API_KEY"]
NSRDB_KEY  = os.environ["NSRDB_API_KEY"]
NSRDB_NAME = os.environ["NSRDB_FULL_NAME"]
NSRDB_EMAIL= os.environ["NSRDB_EMAIL"]
NSRDB_AFF  = os.environ["NSRDB_AFFILIATION"]
print("Environment loaded. Keys present:", all([EIA_KEY, NSRDB_KEY, NSRDB_NAME, NSRDB_EMAIL, NSRDB_AFF]))

Environment loaded. Keys present: True


In [2]:
RAW   = Path("data/raw");   RAW.mkdir(parents=True, exist_ok=True)
CLEAN = Path("data/clean"); CLEAN.mkdir(parents=True, exist_ok=True)

REGIONS = {
    "Virginia": dict(lat=38.95, lon=-77.45, tz="America/New_York",
                     years=[2015,2016,2017], state="VA"),
    "Ohio":     dict(lat=39.96, lon=-82.99, tz="America/New_York",
                     years=[2015,2016,2017], state="OH"),
    "Texas":    dict(lat=32.78, lon=-96.80, tz="America/Chicago",
                     years=[2019,2020,2021], state="TX"),
    "Georgia":  dict(lat=33.75, lon=-84.39, tz="America/New_York",
                     years=[2019,2020,2021], state="GA"),
}

EIA_RESPONDENTS = {"Texas": "ERCO", "Georgia": "SOCO"}

# Expected rows = elapsed hours in each region's LOCAL calendar years (26,304 for 3 years incl. one leap year)
EXPECTED_ROWS = {
    r: int((pd.Timestamp(f"{c['years'][-1] + 1}-01-01", tz=c["tz"])
            - pd.Timestamp(f"{c['years'][0]}-01-01", tz=c["tz"])) / pd.Timedelta(hours=1))
    for r, c in REGIONS.items()
}
print("Config OK. Expected rows:", EXPECTED_ROWS)

Config OK. Expected rows: {'Virginia': 26304, 'Ohio': 26304, 'Texas': 26304, 'Georgia': 26304}


## 1. Data Collection

Four sources:
1. **Virginia** — PJM DOM_hourly.csv (Kaggle, already in project root)
2. **Ohio** — PJM AEP_hourly.csv.zip (Kaggle, already in project root)
3. **Texas / Georgia** — EIA-930 hourly demand API (UTC, hour-ending timestamps)
4. **Weather / Solar** — NREL NSRDB API, one year per call (multi-year returns 400), 30-min → resampled hourly

The analysis uses **local** calendar years. A local year ends 5–6 hours after the UTC year,
so the first hours of the following UTC year are also fetched (EIA-930: a small boundary request;
NSRDB: the following year, because the API only serves whole years). Those extra hours are used
only to complete the last local day; nothing outside the configured local years is kept.

In [3]:
# Extract AEP zip to data/raw (do not modify original)
aep_raw = RAW / "AEP_hourly.csv"
if not aep_raw.exists():
    with zipfile.ZipFile("AEP_hourly.csv.zip") as z:
        name = z.namelist()[0]
        with z.open(name) as src, open(aep_raw, "wb") as dst:
            dst.write(src.read())
    print(f"Extracted → {aep_raw}")
else:
    print(f"Already extracted: {aep_raw}")

Already extracted: data/raw/AEP_hourly.csv


In [4]:
def fetch_eia930(respondent, start, end, cache, api_key):
    """Fetch hourly demand (type D) for one respondent between two UTC period labels
    (inclusive). Returns a list of record dicts and caches them as JSON in data/raw/."""
    if cache.exists():
        with open(cache) as f:
            return json.load(f)

    url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"
    all_rows, offset, page_size = [], 0, 5000
    while True:
        params = {
            "api_key": api_key,
            "frequency": "hourly",
            "data[0]": "value",
            "facets[respondent][]": respondent,
            "facets[type][]": "D",
            "start": start,
            "end": end,
            "sort[0][column]": "period",
            "sort[0][direction]": "asc",
            "length": page_size,
            "offset": offset,
        }
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        batch = r.json()["response"]["data"]
        all_rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
        time.sleep(0.2)

    with open(cache, "w") as f:
        json.dump(all_rows, f)
    return all_rows


# Fetch Texas (ERCO) and Georgia (SOCO): one request per UTC year, plus the first
# hours of the next UTC year so the last LOCAL day is complete.
eia_raw = {}
for region, respondent in EIA_RESPONDENTS.items():
    rows = []
    for yr in REGIONS[region]["years"]:
        batch = fetch_eia930(respondent, f"{yr}-01-01T00", f"{yr}-12-31T23",
                             RAW / f"eia930_{respondent}_{yr}.json", EIA_KEY)
        rows.extend(batch)
        print(f"  {region} {yr}: {len(batch):,} rows fetched/cached")
    nxt = REGIONS[region]["years"][-1] + 1
    batch = fetch_eia930(respondent, f"{nxt}-01-01T00", f"{nxt}-01-01T08",
                         RAW / f"eia930_{respondent}_{nxt}_boundary.json", EIA_KEY)
    rows.extend(batch)
    print(f"  {region} {nxt} boundary hours: {len(batch):,} rows fetched/cached")
    eia_raw[region] = rows
print("EIA-930 collection complete.")

  Texas 2019: 8,760 rows fetched/cached
  Texas 2020: 8,784 rows fetched/cached
  Texas 2021: 8,759 rows fetched/cached
  Texas 2022 boundary hours: 9 rows fetched/cached
  Georgia 2019: 8,759 rows fetched/cached
  Georgia 2020: 8,784 rows fetched/cached
  Georgia 2021: 8,760 rows fetched/cached
  Georgia 2022 boundary hours: 9 rows fetched/cached
EIA-930 collection complete.


In [5]:
def fetch_nsrdb_year(region_name, lat, lon, year, api_key, email, full_name, affiliation):
    """Fetch NSRDB 30-min solar/weather data for one region and one year.
    Caches raw CSV text (with 2 metadata header lines) to data/raw/.
    Returns DataFrame with columns Year,Month,Day,Hour,Minute,GHI,DNI,DHI,Temperature,Wind Speed."""
    cache = RAW / f"nsrdb_{region_name}_{year}.csv"
    if cache.exists():
        df = pd.read_csv(cache, skiprows=2)
        return df

    url = ("https://developer.nlr.gov/api/nsrdb/v2/solar/"
           "nsrdb-GOES-aggregated-v4-0-0-download.csv")
    params = {
        "api_key": api_key,
        "wkt": f"POINT({lon} {lat})",
        "names": str(year),
        "attributes": "ghi,dni,dhi,air_temperature,wind_speed",
        "interval": "30",
        "utc": "true",
        "leap_day": "true",
        "full_name": full_name,
        "email": email,
        "affiliation": affiliation,
        "reason": "Academic research and data analytics project",
        "mailing_list": "false",
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    cache.write_text(r.text, encoding="utf-8")
    df = pd.read_csv(io.StringIO(r.text), skiprows=2)
    return df


# Fetch NSRDB for all 4 regions (one call per year). The year after the last analysis year
# is fetched only to cover the final 5-6 local hours; the merge discards the rest.
nsrdb_raw = {}
for region, cfg in REGIONS.items():
    frames = []
    for yr in cfg["years"] + [cfg["years"][-1] + 1]:
        df = fetch_nsrdb_year(
            region, cfg["lat"], cfg["lon"], yr,
            NSRDB_KEY, NSRDB_EMAIL, NSRDB_NAME, NSRDB_AFF
        )
        frames.append(df)
        print(f"  {region} {yr}: {len(df):,} rows (30-min)")
        time.sleep(0.5)          # be polite to the API
    nsrdb_raw[region] = pd.concat(frames, ignore_index=True)

print("NSRDB collection complete.")

  Virginia 2015: 17,520 rows (30-min)


  Virginia 2016: 17,568 rows (30-min)


  Virginia 2017: 17,520 rows (30-min)


  Virginia 2018: 17,520 rows (30-min)


  Ohio 2015: 17,520 rows (30-min)


  Ohio 2016: 17,568 rows (30-min)


... (7 outputs left out)

  Georgia 2020: 17,568 rows (30-min)


  Georgia 2021: 17,520 rows (30-min)


  Georgia 2022: 17,520 rows (30-min)


NSRDB collection complete.


## 2. Cleaning and Integration

### UTC Alignment Strategy

**Common convention:** `Timestamp` is the UTC **start** of each one-hour interval.
`Hour = 0` therefore means 00:00–01:00 local time.

| Source | Label semantics (evidence) | Conversion to interval start (UTC) |
|--------|---------------------------|------------------------------------|
| PJM (DOM/AEP) | **Hour-ending**, Eastern prevailing time. Evidence in the file: every November the `02:00` label appears twice (PJM's repeated HE02) and every March `03:00` is missing (skipped HE03). Rows are not sorted. | Keep labels Jan 1 01:00 … Jan 1 (next year) 00:00 for each local year; first interval starts Jan 1 00:00 EST = 05:00 UTC; each following row is one elapsed hour. Every unambiguous row is cross-checked against the DST-aware conversion `label − 1 h`. |
| EIA-930 | **Hour-ending, UTC** (EIA-930 instructions: hourly values are reported by hour-ending time with UTC stamps). | `period − 1 h` |
| NSRDB | 30-min values labeled at the **start** of each half hour (checked empirically: the mean GHI curve is centred about 15 min before theoretical solar noon). | Mean of the `:00` and `:30` values, labeled at the hour start. |

All sources are merged on this `Timestamp` and then restricted to each region's local calendar years.

In [6]:
def process_pjm(csv_path, mw_col, region_name, years):
    """
    Load a PJM hourly CSV and convert its hour-ENDING Eastern labels to UTC interval starts.

    - Label 01:00 is the hour 00:00-01:00; label 00:00 is the last hour of the previous day.
    - Local year Y = labels Jan 1 Y 01:00 ... Jan 1 Y+1 00:00 (inclusive).
    - The first interval starts Jan 1 00:00 EST = 05:00 UTC; each row is one elapsed hour,
      so Timestamp = 05:00 UTC + row number. This handles the repeated November label
      (both rows are real hours, kept in file order by a stable sort) and the skipped March label.
    - Every row whose start time is unambiguous is cross-checked against a DST-aware
      conversion of (label - 1 h).
    """
    df = pd.read_csv(csv_path)
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values("Datetime", kind="stable").reset_index(drop=True)

    frames = []
    for yr in years:
        lo = pd.Timestamp(f"{yr}-01-01 01:00")
        hi = pd.Timestamp(f"{yr+1}-01-01 00:00")
        yr_df = df[(df["Datetime"] >= lo) & (df["Datetime"] <= hi)].copy().reset_index(drop=True)
        assert yr_df["Datetime"].iloc[0] == lo and yr_df["Datetime"].iloc[-1] == hi, (
            f"{region_name} {yr}: labels do not span {lo} to {hi}")

        start_utc = pd.Timestamp(f"{yr}-01-01 05:00", tz="UTC")
        yr_df["Timestamp"] = pd.date_range(start=start_utc, periods=len(yr_df), freq="h")

        # Row count check: last interval must start Dec 31 23:00 EST = Jan 1 04:00 UTC
        expected_last = pd.Timestamp(f"{yr+1}-01-01 04:00", tz="UTC")
        assert yr_df["Timestamp"].iloc[-1] == expected_last, (
            f"{region_name} {yr}: last UTC={yr_df['Timestamp'].iloc[-1]}, expected {expected_last}")

        # Per-row check against the DST-aware conversion (ambiguous November hour skipped)
        start_local = (yr_df["Datetime"] - pd.Timedelta(hours=1)).dt.tz_localize(
            "America/New_York", ambiguous="NaT", nonexistent="NaT")
        ok = start_local.isna() | (start_local.dt.tz_convert("UTC") == yr_df["Timestamp"])
        assert ok.all(), f"{region_name} {yr}: {(~ok).sum()} rows disagree with DST conversion"

        yr_df["Region"] = region_name
        yr_df["Demand_MW"] = yr_df[mw_col].astype(float)
        frames.append(yr_df[["Region", "Timestamp", "Demand_MW"]])
        print(f"  {region_name} {yr}: {len(yr_df):,} rows, "
              f"UTC {yr_df['Timestamp'].iloc[0]} → {yr_df['Timestamp'].iloc[-1]}")

    return pd.concat(frames, ignore_index=True)


demand_va = process_pjm("DOM_hourly.csv",    "DOM_MW", "Virginia", [2015,2016,2017])
demand_oh = process_pjm(str(RAW/"AEP_hourly.csv"), "AEP_MW", "Ohio", [2015,2016,2017])
print(f"Virginia rows: {len(demand_va):,}  Ohio rows: {len(demand_oh):,}")

  Virginia 2015: 8,760 rows, UTC 2015-01-01 05:00:00+00:00 → 2016-01-01 04:00:00+00:00
  Virginia 2016: 8,784 rows, UTC 2016-01-01 05:00:00+00:00 → 2017-01-01 04:00:00+00:00
  Virginia 2017: 8,760 rows, UTC 2017-01-01 05:00:00+00:00 → 2018-01-01 04:00:00+00:00
  Ohio 2015: 8,760 rows, UTC 2015-01-01 05:00:00+00:00 → 2016-01-01 04:00:00+00:00
  Ohio 2016: 8,784 rows, UTC 2016-01-01 05:00:00+00:00 → 2017-01-01 04:00:00+00:00
  Ohio 2017: 8,760 rows, UTC 2017-01-01 05:00:00+00:00 → 2018-01-01 04:00:00+00:00
Virginia rows: 26,304  Ohio rows: 26,304


In [7]:
def process_eia930(raw_rows, region_name, tz, years):
    """Convert EIA-930 records to a demand DataFrame keyed by UTC interval start.

    `period` is the UTC hour ENDING time, so the interval start is period - 1 h.
    `value` is in MWh for a one-hour interval, which equals the average MW in that hour.
    Only hours that start inside the region's local calendar years are kept."""
    df = pd.DataFrame(raw_rows)
    df["Timestamp"] = (pd.to_datetime(df["period"], format="%Y-%m-%dT%H", utc=True)
                       - pd.Timedelta(hours=1))
    df["Demand_MW"] = pd.to_numeric(df["value"], errors="coerce")
    df["Region"] = region_name
    df = df[df["Timestamp"].dt.tz_convert(tz).dt.year.isin(years)]
    df = df[["Region", "Timestamp", "Demand_MW"]].sort_values("Timestamp").reset_index(drop=True)
    assert not df["Timestamp"].duplicated().any(), f"{region_name}: duplicate periods"

    # Report hours the API did not return, and null values (dropped, never filled)
    full = pd.date_range(pd.Timestamp(f"{years[0]}-01-01", tz=tz),
                         pd.Timestamp(f"{years[-1] + 1}-01-01", tz=tz),
                         freq="h", inclusive="left").tz_convert("UTC")
    missing = full.difference(df["Timestamp"])
    print(f"  {region_name}: {len(missing)} hours not returned by the API: "
          f"{[str(t) for t in missing[:5]]}{' ...' if len(missing) > 5 else ''}")
    nulls = df["Demand_MW"].isnull().sum()
    if nulls > 0:
        null_days = sorted(df.loc[df["Demand_MW"].isnull(), "Timestamp"]
                           .dt.tz_convert(tz).dt.date.astype(str).unique())
        print(f"  {region_name}: {nulls} null Demand_MW values dropped (local dates {null_days})")
        df = df.dropna(subset=["Demand_MW"]).reset_index(drop=True)

    print(f"  {region_name}: {len(df):,} rows, "
          f"UTC {df['Timestamp'].iloc[0]} → {df['Timestamp'].iloc[-1]}")
    return df


demand_tx = process_eia930(eia_raw["Texas"],   "Texas",   REGIONS["Texas"]["tz"],   REGIONS["Texas"]["years"])
demand_ga = process_eia930(eia_raw["Georgia"], "Georgia", REGIONS["Georgia"]["tz"], REGIONS["Georgia"]["years"])
print(f"Texas rows: {len(demand_tx):,}  Georgia rows: {len(demand_ga):,}")

  Texas: 1 hours not returned by the API: ['2021-11-07 05:00:00+00:00']
  Texas: 26,303 rows, UTC 2019-01-01 06:00:00+00:00 → 2022-01-01 05:00:00+00:00
  Georgia: 1 hours not returned by the API: ['2019-11-04 05:00:00+00:00']
  Georgia: 24 null Demand_MW values dropped (local dates ['2019-11-03'])
  Georgia: 26,279 rows, UTC 2019-01-01 05:00:00+00:00 → 2022-01-01 04:00:00+00:00
Texas rows: 26,303  Georgia rows: 26,279


In [8]:
def process_nsrdb(raw_df, region_name):
    """
    Convert 30-min NSRDB DataFrame to hourly UTC.
    Build Timestamp from Year/Month/Day/Hour/Minute (UTC), then take the hourly mean of
    the :00 and :30 values, labeled at the hour start (same convention as demand).
    """
    df = raw_df.copy()
    df["Timestamp"] = pd.to_datetime(
        df[["Year","Month","Day","Hour","Minute"]]
        .rename(columns={"Year":"year","Month":"month","Day":"day",
                         "Hour":"hour","Minute":"minute"}),
        utc=True
    )
    df = df.sort_values("Timestamp").reset_index(drop=True)

    df = df.set_index("Timestamp")
    hourly = df[["GHI","DNI","DHI","Temperature","Wind Speed"]].resample("h").mean()
    hourly = hourly.reset_index()
    hourly["Region"] = region_name
    print(f"  {region_name}: {len(raw_df):,} 30-min → {len(hourly):,} hourly rows, "
          f"UTC {hourly['Timestamp'].iloc[0]} → {hourly['Timestamp'].iloc[-1]}")
    return hourly


weather = {}
for region in REGIONS:
    weather[region] = process_nsrdb(nsrdb_raw[region], region)
print("NSRDB processing complete.")

  Virginia: 70,128 30-min → 35,064 hourly rows, UTC 2015-01-01 00:00:00+00:00 → 2018-12-31 23:00:00+00:00
  Ohio: 70,128 30-min → 35,064 hourly rows, UTC 2015-01-01 00:00:00+00:00 → 2018-12-31 23:00:00+00:00
  Texas: 70,128 30-min → 35,064 hourly rows, UTC 2019-01-01 00:00:00+00:00 → 2022-12-31 23:00:00+00:00
  Georgia: 70,128 30-min → 35,064 hourly rows, UTC 2019-01-01 00:00:00+00:00 → 2022-12-31 23:00:00+00:00
NSRDB processing complete.


In [9]:
def merge_region(demand_df, weather_df, region_name, tz_str):
    """Merge demand and weather on UTC Timestamp, add local-time columns."""
    merged = pd.merge(demand_df, weather_df.drop(columns=["Region"]),
                      on="Timestamp", how="inner")

    # Local time derived from UTC — extract features BEFORE converting to string
    local_ts = merged["Timestamp"].dt.tz_convert(tz_str)

    merged["Year"]      = local_ts.dt.year
    merged["Month"]     = local_ts.dt.month
    merged["Hour"]      = local_ts.dt.hour
    merged["DayOfWeek"] = local_ts.dt.dayofweek        # 0=Monday
    merged["Is_Weekend"]= (local_ts.dt.dayofweek >= 5).astype(int)

    def season(m):
        if m in [12,1,2]:  return "Winter"
        if m in [3,4,5]:   return "Spring"
        if m in [6,7,8]:   return "Summer"
        return "Fall"
    merged["Season"] = merged["Month"].map(season)

    # Store as string so pd.concat across different timezones works
    merged["Timestamp_Local"] = local_ts.astype(str)

    print(f"  {region_name}: {len(merged):,} merged rows; "
          f"{len(demand_df) - len(merged):,} demand hours had no matching weather hour")
    return merged


merged_frames = []
for region, cfg in REGIONS.items():
    demand_map = {"Virginia": demand_va, "Ohio": demand_oh,
                  "Texas": demand_tx, "Georgia": demand_ga}
    m = merge_region(demand_map[region], weather[region], region, cfg["tz"])
    merged_frames.append(m)

master = pd.concat(merged_frames, ignore_index=True)
# Ensure Timestamp is UTC tz-aware
master["Timestamp"] = master["Timestamp"].dt.tz_convert("UTC")
print(f"\nMaster shape: {master.shape}")
print("Columns:", master.columns.tolist())

  Virginia: 26,304 merged rows; 0 demand hours had no matching weather hour


  Ohio: 26,304 merged rows; 0 demand hours had no matching weather hour


  Texas: 26,303 merged rows; 0 demand hours had no matching weather hour


  Georgia: 26,279 merged rows; 0 demand hours had no matching weather hour

Master shape: (105190, 15)
Columns: ['Region', 'Timestamp', 'Demand_MW', 'GHI', 'DNI', 'DHI', 'Temperature', 'Wind Speed', 'Year', 'Month', 'Hour', 'DayOfWeek', 'Is_Weekend', 'Season', 'Timestamp_Local']


## 3. Data Quality Checks

Verifying: row counts per region, zero duplicate (Region, Timestamp) pairs, null values after merge, and local hour of minimum demand (expected 3–5 AM local).

In [10]:
print("=" * 60)
print("ROW COUNTS PER REGION")
print("=" * 60)
coverage = master.groupby("Region").size().reset_index(name="n_rows")
coverage["expected"] = coverage["Region"].map(EXPECTED_ROWS)
coverage["gap"] = coverage["n_rows"] - coverage["expected"]
coverage["pct_coverage"] = (coverage["n_rows"] / coverage["expected"] * 100).round(2)
print(coverage.to_string(index=False))
for _, row in coverage.iterrows():
    if row["gap"] < 0:
        print(f"  *** WARN: {row['Region']} is {-row['gap']:,} rows short of full coverage ***")

print()
print("=" * 60)
print("DUPLICATE (Region, Timestamp) CHECK")
print("=" * 60)
dupes = master.duplicated(subset=["Region","Timestamp"]).sum()
print(f"Duplicate pairs: {dupes}  (must be 0)")
assert dupes == 0, f"FAIL: {dupes} duplicates found!"
print("PASS: zero duplicates")

print()
print("=" * 60)
print("NULL CHECK AFTER MERGE")
print("=" * 60)
nulls = master.isnull().sum()
nulls_nonzero = nulls[nulls > 0]
if len(nulls_nonzero) == 0:
    print("No nulls found.")
else:
    print(nulls_nonzero.to_string())

print()
print("=" * 60)
print("TOTAL ROWS:", f"{len(master):,}")
if len(master) < 100_000:
    raise RuntimeError(f"FAIL: master has {len(master):,} rows, minimum is 100,000")

ROW COUNTS PER REGION
  Region  n_rows  expected  gap  pct_coverage
 Georgia   26279     26304  -25         99.90
    Ohio   26304     26304    0        100.00
   Texas   26303     26304   -1        100.00
Virginia   26304     26304    0        100.00
  *** WARN: Georgia is 25 rows short of full coverage ***
  *** WARN: Texas is 1 rows short of full coverage ***

DUPLICATE (Region, Timestamp) CHECK
Duplicate pairs: 0  (must be 0)
PASS: zero duplicates

NULL CHECK AFTER MERGE
No nulls found.

TOTAL ROWS: 105,190


In [11]:
print("=" * 60)
print("CLOCK ALIGNMENT CHECK: Hour of Daily Minimum Demand")
print("Expected: 3-5 AM local for all regions")
print("=" * 60)
# Group by LOCAL date; Hour is the local start hour of each interval
local_date = master["Timestamp_Local"].str[:10]
min_hour_idx = master.groupby([master["Region"], local_date])["Demand_MW"].idxmin()
min_hour_vals = master.loc[min_hour_idx, ["Region", "Hour"]]
mode_hour = min_hour_vals.groupby("Region")["Hour"].agg(lambda x: x.mode().iloc[0])
print(mode_hour.to_string())
for region, h in mode_hour.items():
    if not (3 <= h <= 5):
        print(f"  *** INVESTIGATE: {region} mode min-demand hour = {h:02d}:00 (expected 03-05 local) ***")
    else:
        print(f"  OK: {region} mode min-demand hour = {h:02d}:00 local")

CLOCK ALIGNMENT CHECK: Hour of Daily Minimum Demand
Expected: 3-5 AM local for all regions
Region
Georgia     4
Ohio        3
Texas       3
Virginia    3
  OK: Georgia mode min-demand hour = 04:00 local
  OK: Ohio mode min-demand hour = 03:00 local
  OK: Texas mode min-demand hour = 03:00 local
  OK: Virginia mode min-demand hour = 03:00 local


## 4. Feature Engineering

### Stress Index

`Stress_Index` = percentile rank (0–100) of `Demand_MW` **within each region's own data**:

`Stress_Index = (rank − 1) / (n − 1) × 100`, where `rank` is the average rank of the hour's demand
among the region's `n` hours (ties share their average rank). The lowest hour scores 0 and the highest 100.

- It is a **relative, descriptive** measure, not an engineering or capacity threshold.
- A value of 95 means that hour's demand is in the top 5 % of that region's history.
- Comparisons **across regions** are not meaningful because each region has its own baseline.

In [12]:
def stress_index(series):
    """Percentile rank within the series, 0–100: (average rank - 1) / (n - 1) * 100."""
    ranks = series.rank(method="average")
    return (ranks - 1) / (len(ranks) - 1) * 100

master["Stress_Index"] = (
    master.groupby("Region")["Demand_MW"]
    .transform(stress_index)
)

print("Stress_Index stats per region:")
print(master.groupby("Region")["Stress_Index"].describe().round(2).to_string())
print()
print("Master schema:")
print(master.dtypes.to_string())

Stress_Index stats per region:
             count  mean   std  min   25%   50%   75%    max
Region                                                      
Georgia  26,279.00 50.00 28.87 0.00 25.00 50.01 75.00 100.00
Ohio     26,304.00 50.00 28.87 0.00 25.00 50.01 75.01 100.00
Texas    26,303.00 50.00 28.87 0.00 25.00 50.00 75.00 100.00
Virginia 26,304.00 50.00 28.87 0.00 24.99 49.99 75.00 100.00

Master schema:
Region                             str
Timestamp          datetime64[us, UTC]
Demand_MW                      float64
GHI                            float64
DNI                            float64
DHI                            float64
Temperature                    float64
Wind Speed                     float64
Year                             int32
Month                            int32
Hour                             int32
DayOfWeek                        int32
Is_Weekend                       int64
Season                             str
Timestamp_Local                    str
St

## 5. IM3 Data Center Atlas

**Source:** PNNL / DOE MSD-LIVE, record p147s-4h760 v2026.02.09, DOI 10.57931/3017294, ODbL license.

**Layer types:**
- `point` — location only, no `sqft`
- `building` — single structure, has `sqft`
- `campus` — multi-building facility, has `sqft`

Buildings and campuses overlap at the same facility, so we **never sum `sqft` across types**. Unique data centers are counted with `nunique(id)`.

`sqft` is the facility polygon area in **square feet** (IM3 README). `point` records have no area;
it stays missing (never imputed). The atlas is a **February 2026 snapshot**, while the demand data
covers 2015–2021, so some facilities did not exist during the demand period.

Two files are written: `im3_atlas_full.csv` (all states, coordinates required; non-focus states
labeled `Other`, used for the dashboard's national context map) and `im3_clean.csv` (the four
focus states only).

In [13]:
IM3_PATH = (
    "p147s-4h760/im3_open_source_data_center_atlas_v2026.02.09/"
    "im3_open_source_data_center_atlas_v2026.02.09.csv"
)
im3_raw = pd.read_csv(IM3_PATH)
print(f"IM3 raw: {im3_raw.shape[0]:,} rows, columns: {im3_raw.columns.tolist()}")
print("Type counts:")
print(im3_raw["type"].value_counts().to_string())

# All states with coordinates; focus states get their Region name, the rest "Other"
inv_state_map = {cfg["state"]: region for region, cfg in REGIONS.items()}
im3_full = im3_raw.dropna(subset=["lat","lon"]).copy()
im3_full["Region"] = im3_full["state_abb"].map(inv_state_map).fillna("Other")
im3_clean = im3_full[im3_full["Region"] != "Other"].copy()
print(f"\nIM3 with coordinates (all states): {len(im3_full):,} rows, "
      f"{im3_full['id'].nunique():,} unique ids, {im3_full['state_abb'].nunique()} states")
print(f"IM3 filtered (4 focus states): {len(im3_clean):,} rows")
print(im3_clean.groupby("Region")["id"].nunique().rename("unique_DCs").to_string())
print("sqft nulls per region+type:")
print(im3_clean.groupby(["Region","type"])["sqft"].apply(lambda x: x.isnull().sum()).to_string())

IM3 raw: 1,479 rows, columns: ['id', 'state', 'state_abb', 'state_id', 'county', 'county_id', 'operator', 'ref', 'name', 'sqft', 'lon', 'lat', 'type']
Type counts:
type
building    1239
campus       135
point        105

IM3 with coordinates (all states): 1,479 rows, 1,474 unique ids, 47 states
IM3 filtered (4 focus states): 577 rows
Region
Georgia      37
Ohio         91
Texas       127
Virginia    319
sqft nulls per region+type:
Region    type    
Georgia   building    0
          campus      0
          point       3
Ohio      building    0
          campus      0
          point       1
Texas     building    0
          campus      0
          point       4
Virginia  building    0
          campus      0
          point       7


In [14]:
# geo_summary: one row per Region
# sqft reported separately per layer type to avoid double-counting
demand_summary = (
    master.groupby("Region")["Demand_MW"]
    .agg(avg_demand_MW="mean", peak_demand_MW="max", std_demand_MW="std")
    .round(1)
)

def sqft_by_type(grp, layer_type):
    sub = grp[grp["type"] == layer_type]
    # Campus and building: sum sqft of UNIQUE ids only (multi-county rows share same id)
    unique_sqft = sub.drop_duplicates("id")["sqft"].sum()
    return unique_sqft

geo_rows = []
for region in REGIONS:
    grp = im3_clean[im3_clean["Region"] == region]
    geo_rows.append({
        "Region": region,
        "unique_DCs": grp["id"].nunique(),
        "n_building": grp[grp["type"]=="building"]["id"].nunique(),
        "n_campus":   grp[grp["type"]=="campus"]["id"].nunique(),
        "n_point":    grp[grp["type"]=="point"]["id"].nunique(),
        "building_area_sqft": sqft_by_type(grp, "building"),   # square feet
        "campus_area_sqft":   sqft_by_type(grp, "campus"),     # square feet
    })

geo_summary = (
    pd.DataFrame(geo_rows)
    .merge(demand_summary.reset_index(), on="Region")
)
print("geo_summary:")
print(geo_summary.to_string(index=False))

geo_summary:
  Region  unique_DCs  n_building  n_campus  n_point  building_area_sqft  campus_area_sqft  avg_demand_MW  peak_demand_MW  std_demand_MW
Virginia         319         281        31        7       41,783,929.00     56,960,153.00      11,117.10       21,651.00       2,489.10
    Ohio          91          81         9        1       15,373,043.00     63,941,146.00      14,712.40       24,739.00       2,503.90
   Texas         127         113        10        4       18,888,418.00     21,323,769.00      43,994.00       74,533.00       9,537.80
 Georgia          37          27         7        3        6,618,529.00     15,577,999.00      26,574.20       45,515.00       5,469.80


## 6. Exploratory Data Analysis

All charts include title, axis labels with units, and a one-line takeaway derived from the real data.

In [15]:
print("DESCRIPTIVE STATISTICS — Demand_MW per Region")
stats = master.groupby("Region")["Demand_MW"].describe().round(1)
print(stats.to_string())
print()
print("DESCRIPTIVE STATISTICS — Temperature (°C) per Region")
print(master.groupby("Region")["Temperature"].describe().round(1).to_string())

DESCRIPTIVE STATISTICS — Demand_MW per Region
             count      mean      std       min       25%       50%       75%       max
Region                                                                                 
Georgia  26,279.00 26,574.20 5,469.80  6,913.00 22,805.50 25,322.00 29,426.00 45,515.00
Ohio     26,304.00 14,712.40 2,503.90  9,581.00 12,876.80 14,375.00 16,283.00 24,739.00
Texas    26,303.00 43,994.00 9,537.80 27,449.00 37,294.50 41,449.00 48,697.00 74,533.00
Virginia 26,304.00 11,117.10 2,489.10  6,598.00  9,408.00 10,572.00 12,590.00 21,651.00

DESCRIPTIVE STATISTICS — Temperature (°C) per Region
             count  mean   std    min   25%   50%   75%   max
Region                                                       
Georgia  26,279.00 16.90  9.20  -6.80  9.90 17.70 23.30 39.00
Ohio     26,304.00 11.90 11.10 -19.00  3.00 13.20 20.90 35.30
Texas    26,303.00 19.10  9.80 -15.40 11.70 20.00 26.30 42.60
Virginia 26,304.00 13.40 10.90 -19.00  4.80 14.20 22.00 38.00


In [16]:
print("PEAK AND MINIMUM DEMAND WITH TIMESTAMPS")
for region in REGIONS:
    sub = master[master["Region"] == region]
    peak_idx = sub["Demand_MW"].idxmax()
    min_idx  = sub["Demand_MW"].idxmin()
    peak_row = sub.loc[peak_idx]
    min_row  = sub.loc[min_idx]
    print(f"\n{region}:")
    print(f"  Peak: {peak_row['Demand_MW']:,.0f} MW  at {peak_row['Timestamp_Local']} local")
    print(f"  Min:  {min_row['Demand_MW']:,.0f} MW  at {min_row['Timestamp_Local']} local")

PEAK AND MINIMUM DEMAND WITH TIMESTAMPS



Virginia:
  Peak: 21,651 MW  at 2015-02-20 07:00:00-05:00 local
  Min:  6,598 MW  at 2016-10-09 04:00:00-04:00 local

Ohio:
  Peak: 24,739 MW  at 2015-02-20 07:00:00-05:00 local
  Min:  9,581 MW  at 2016-10-02 04:00:00-04:00 local

Texas:
  Peak: 74,533 MW  at 2019-08-12 16:00:00-05:00 local
  Min:  27,449 MW  at 2020-05-10 05:00:00-05:00 local

Georgia:
  Peak: 45,515 MW  at 2019-08-13 16:00:00-04:00 local
  Min:  6,913 MW  at 2019-10-25 23:00:00-04:00 local


In [17]:
# Daily averages use the LOCAL calendar date
daily_avg = (
    master.assign(Date=pd.to_datetime(master["Timestamp_Local"].str[:10]))
    .groupby(["Region", "Date"])["Demand_MW"].mean().reset_index()
)

fig = px.line(
    daily_avg, x="Date", y="Demand_MW", facet_col="Region",
    facet_col_wrap=2, height=500,
    title="Daily Average Electricity Demand by Region",
    labels={"Demand_MW": "Demand (MW)", "Date": "Date"},
    color="Region",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# Takeaway
top_day = daily_avg.loc[daily_avg["Demand_MW"].idxmax()]
monthly_means = master.groupby(["Region", "Month"])["Demand_MW"].mean()
swing = (monthly_means.groupby("Region").max() / monthly_means.groupby("Region").min())
swing_str = ", ".join(f"{r} {v:.2f}x" for r, v in swing.items())
takeaway = (f"Takeaway: highest daily average is {top_day['Region']} "
            f"({top_day['Demand_MW']:,.0f} MW on {top_day['Date']:%Y-%m-%d}). "
            f"Highest/lowest monthly mean ratio: {swing_str}.")
fig.add_annotation(text=takeaway, xref="paper", yref="paper",
                   x=0, y=-0.12, showarrow=False, font=dict(size=11, color="gray"))
fig.show()

In [18]:
hourly_profile = (
    master.groupby(["Region","Hour","Is_Weekend"])["Demand_MW"]
    .mean().reset_index()
)
hourly_profile["Day Type"] = hourly_profile["Is_Weekend"].map({0:"Weekday",1:"Weekend"})

fig = px.line(
    hourly_profile, x="Hour", y="Demand_MW", color="Day Type",
    facet_col="Region", facet_col_wrap=2, height=500,
    title="Average Hourly Demand Profile: Weekday vs Weekend",
    labels={"Demand_MW": "Avg Demand (MW)", "Hour": "Hour of Day (local)"},
    color_discrete_map={"Weekday":"steelblue","Weekend":"coral"},
)
fig.update_xaxes(dtick=3)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# Compute takeaway from data
wkday = hourly_profile[hourly_profile["Is_Weekend"]==0].groupby("Region")["Demand_MW"].mean()
wkend = hourly_profile[hourly_profile["Is_Weekend"]==1].groupby("Region")["Demand_MW"].mean()
diff_pct = ((wkday - wkend) / wkend * 100).mean()
takeaway = (f"Takeaway: Weekday demand averages {abs(diff_pct):.1f}% "
            f"{'higher' if diff_pct > 0 else 'lower'} than weekends (mean of the four regions' "
            f"hourly-profile averages).")
fig.add_annotation(text=takeaway, xref="paper", yref="paper",
                   x=0, y=-0.12, showarrow=False, font=dict(size=11, color="gray"))
fig.show()

In [19]:
monthly_avg = (
    master.groupby(["Region","Month"])["Demand_MW"].mean().reset_index()
)
monthly_avg["Month_Name"] = pd.to_datetime(monthly_avg["Month"], format="%m").dt.strftime("%b")

fig = px.bar(
    monthly_avg, x="Month_Name", y="Demand_MW", color="Region",
    barmode="group", height=450,
    title="Average Monthly Demand by Region",
    labels={"Demand_MW": "Avg Demand (MW)", "Month_Name": "Month"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    category_orders={"Month_Name":["Jan","Feb","Mar","Apr","May","Jun",
                                    "Jul","Aug","Sep","Oct","Nov","Dec"]},
)
# Takeaway
peak_months = monthly_avg.loc[monthly_avg.groupby("Region")["Demand_MW"].idxmax(), ["Region", "Month_Name"]]
takeaway = "Takeaway: highest-demand month by region: " + ", ".join(
    f"{r} {m}" for r, m in peak_months.itertuples(index=False)) + "."
fig.add_annotation(text=takeaway, xref="paper", yref="paper",
                   x=0, y=-0.15, showarrow=False, font=dict(size=11, color="gray"))
fig.show()

# Seasonal box plot
fig2 = px.box(
    master, x="Season", y="Demand_MW", color="Region",
    height=450,
    title="Demand Distribution by Season",
    labels={"Demand_MW": "Demand (MW)", "Season": "Season"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    category_orders={"Season":["Winter","Spring","Summer","Fall"]},
)
fig2.show()

In [20]:
# Random 1/6 sample per region (fixed seed) to keep the plot light; every hour of day is represented.
# Correlations below use ALL hours, not the sample.
sample = master.groupby("Region").sample(frac=1/6, random_state=42)

fig = px.scatter(
    sample, x="Temperature", y="Demand_MW",
    color="Region", facet_col="Region", facet_col_wrap=2,
    opacity=0.3, height=550, trendline="lowess",
    title="Temperature vs Demand (hourly sample, LOWESS trend)",
    labels={"Temperature":"Temperature (°C)","Demand_MW":"Demand (MW)"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_traces(marker=dict(size=3), selector=dict(mode="markers"))

# Compute correlations
corrs = {r: g["Temperature"].corr(g["Demand_MW"]) for r, g in master.groupby("Region")}
corr_str = ", ".join(f"{r}:{v:+.2f}" for r, v in corrs.items())
takeaway = (f"Takeaway: Pearson r (temperature vs demand, all hours) — {corr_str}. "
            "r measures linear association only; the LOWESS curve shows the shape.")
fig.add_annotation(text=takeaway, xref="paper", yref="paper",
                   x=0, y=-0.12, showarrow=False, font=dict(size=10, color="gray"))
fig.show()

In [21]:
# GHI vs demand: daytime hours only (GHI > 0). Plot a random 1/4 sample per region;
# correlations use all daytime hours.
daytime = master[master["GHI"] > 0]
daytime_sample = daytime.groupby("Region").sample(frac=1/4, random_state=42)

fig = px.scatter(
    daytime_sample, x="GHI", y="Demand_MW",
    color="Region", facet_col="Region", facet_col_wrap=2,
    opacity=0.25, height=500,
    title="Solar Irradiance (GHI) vs Demand — Daytime Hours",
    labels={"GHI":"GHI (W/m²)","Demand_MW":"Demand (MW)"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_traces(marker=dict(size=3))

ghi_corrs = {r: g["GHI"].corr(g["Demand_MW"]) for r, g in daytime.groupby("Region")}
ghi_str = ", ".join(f"{r}:{v:+.2f}" for r, v in ghi_corrs.items())
takeaway = f"Takeaway: Pearson r (GHI vs demand, all daytime hours) — {ghi_str}."
fig.add_annotation(text=takeaway, xref="paper", yref="paper",
                   x=0, y=-0.12, showarrow=False, font=dict(size=10, color="gray"))
fig.show()

In [22]:
fig = px.histogram(
    master, x="Stress_Index", color="Region",
    facet_col="Region", facet_col_wrap=2,
    nbins=50, height=450,
    title="Stress Index Distribution per Region (0–100 percentile rank)",
    labels={"Stress_Index":"Stress Index (percentile rank)"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(showlegend=False)

top5_count = (master["Stress_Index"] >= 95).sum()
takeaway = (f"Takeaway: {top5_count:,} hours ({top5_count/len(master)*100:.1f}% of all) "
            f"have Stress Index ≥ 95 (top 5% per region). "
            f"Distribution is approximately uniform by construction.")
fig.add_annotation(text=takeaway, xref="paper", yref="paper",
                   x=0, y=-0.12, showarrow=False, font=dict(size=10, color="gray"))
fig.show()

In [23]:
for region in REGIONS:
    sub = master[master["Region"] == region]
    heat = sub.groupby(["Month","Hour"])["Stress_Index"].mean().reset_index()
    heat_pivot = heat.pivot(index="Hour", columns="Month", values="Stress_Index")

    month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"]

    fig = px.imshow(
        heat_pivot, aspect="auto", color_continuous_scale="RdYlGn_r",
        title=f"{region}: Mean Stress Index — Hour × Month",
        labels={"x":"Month","y":"Hour of Day (local)","color":"Stress Index"},
        x=[month_labels[m-1] for m in sorted(heat_pivot.columns)],
    )
    peak_m, peak_h = heat.loc[heat["Stress_Index"].idxmax(), ["Month","Hour"]]
    takeaway = (f"Takeaway: Highest mean stress in {month_labels[int(peak_m)-1]}, "
                f"hour {int(peak_h):02d}:00 local.")
    fig.add_annotation(text=takeaway, xref="paper", yref="paper",
                       x=0, y=-0.12, showarrow=False, font=dict(size=11, color="gray"))
    fig.show()

In [24]:
top1 = master[master["Stress_Index"] >= 99].copy()
print(f"Hours at Stress Index ≥ 99 (top 1%): {len(top1):,}")
print()
by_region = top1.groupby("Region").agg(
    n_hours=("Stress_Index","count"),
    peak_demand=("Demand_MW","max"),
    most_common_month=("Month", lambda x: x.mode().iloc[0]),
    most_common_hour=("Hour", lambda x: x.mode().iloc[0]),
).reset_index()
by_region["most_common_month"] = by_region["most_common_month"].apply(
    lambda m: pd.to_datetime(str(m), format="%m").strftime("%B"))
print(by_region.to_string(index=False))

fig = px.histogram(
    top1, x="Month", color="Region", barmode="group",
    facet_col="Region", facet_col_wrap=2, nbins=12,
    title="Top 1% Stress Hours: Distribution by Month",
    labels={"Month":"Month","count":"Number of Hours"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(showlegend=False)
fig.show()

Hours at Stress Index ≥ 99 (top 1%): 1,055

  Region  n_hours  peak_demand most_common_month  most_common_hour
 Georgia      263    45,515.00            August                16
    Ohio      264    24,739.00          February                16
   Texas      264    74,533.00            August                16
Virginia      264    21,651.00              July                16


In [25]:
# Building/campus records are sized by their real floor area (marker area ∝ sqft).
# Point records have no area: drawn as small black "x" markers, never given a made-up size.
has_area = im3_clean["sqft"].notna()
fig = px.scatter_geo(
    im3_clean[has_area], lat="lat", lon="lon",
    color="Region", symbol="type", size="sqft", size_max=25,
    hover_name="name",
    hover_data={"sqft":":,.0f", "operator":True, "county":True, "lat":False, "lon":False},
    scope="usa", height=550,
    title="IM3 Data Center Atlas — Virginia, Ohio, Texas, Georgia (marker area ∝ floor area, sq ft)",
    labels={"Region":"Region","type":"Layer Type","sqft":"Floor area (sq ft)"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
pts = im3_clean[~has_area]
fig.add_trace(go.Scattergeo(
    lat=pts["lat"], lon=pts["lon"], mode="markers", name="point (no floor area)",
    marker=dict(symbol="x", size=6, color="black"), text=pts["name"],
    hovertemplate="%{text}<br>point record, floor area not available<extra></extra>",
))
fig.update_geos(fitbounds="locations", showland=True, landcolor="lightgray",
                showsubunits=True, subunitcolor="white")
fig.update_layout(margin=dict(l=0,r=0,t=50,b=0))
dc_counts = im3_clean.groupby("Region")["id"].nunique().sort_values(ascending=False)
takeaway = ("Takeaway: unique data centers — "
            + ", ".join(f"{r} {n}" for r, n in dc_counts.items())
            + f". {pts['id'].nunique()} point records have no floor area (x markers).")
fig.add_annotation(text=takeaway, xref="paper", yref="paper",
                   x=0, y=-0.06, showarrow=False, font=dict(size=11, color="gray"))
fig.show()

In [26]:
fig = px.scatter(
    geo_summary, x="unique_DCs", y="avg_demand_MW",
    color="Region", size="peak_demand_MW",
    text="Region", height=450,
    title="Data Center Count vs Average Demand per Region",
    labels={"unique_DCs":"Unique Data Centers (all types)",
            "avg_demand_MW":"Average Demand (MW)",
            "peak_demand_MW":"Peak Demand (MW, bubble size)"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(textposition="top center")
fig.update_layout(showlegend=False)
top_dc  = geo_summary.loc[geo_summary["unique_DCs"].idxmax()]
top_avg = geo_summary.loc[geo_summary["avg_demand_MW"].idxmax()]
low_avg = geo_summary.loc[geo_summary["avg_demand_MW"].idxmin()]
takeaway = (f"Takeaway: most data centers: {top_dc['Region']} ({top_dc['unique_DCs']}); "
            f"highest average demand: {top_avg['Region']} ({top_avg['avg_demand_MW']:,.0f} MW); "
            f"lowest: {low_avg['Region']} ({low_avg['avg_demand_MW']:,.0f} MW). "
            "Four regions are too few to establish a relationship.")
fig.add_annotation(text=takeaway, xref="paper", yref="paper",
                   x=0, y=-0.12, showarrow=False, font=dict(size=11, color="gray"))
fig.show()

## 7. Synthetic Demonstration Data (generated, NOT measured)

**Why.** For classroom demonstration only, the dashboard can show a larger dataset. The real data alone
already meets the project's size requirement (105,190 hourly rows); the synthetic rows add no evidence and
**every analysis above uses real data only**.

**How they are generated (seed 42, fully reproducible).**
1. **Period:** 5,000 consecutive hours per region (20,000 rows in total), starting immediately after each
   region's real period — Virginia/Ohio from 2018-01-01, Texas/Georgia from 2022-01-01 (local time). They never
   overlap real timestamps. These calendar dates are labels for a synthetic period only: they do **not**
   represent what actually happened on those dates (real data for some of them exists but is not used).
2. **Template:** each region copies one *continuous* stretch of its own real data — a real year chosen with
   the fixed seed — starting on the first date on or after 1 January that falls on the same weekday as the synthetic
   start. Copying one continuous stretch keeps the hourly shape, the weekly rhythm, the season and the joint
   relationships between demand and weather, and creates no artificial jumps at day boundaries. (An earlier
   day-by-day resampling version was rejected because its day seams produced one-hour demand spikes.)
3. **Template check:** the stretch must contain no missing hour and none of the isolated one-hour demand spikes
   (suspected source errors); otherwise another real year is drawn.
4. **Smooth, bounded perturbations** so no row is a copy of a real one: demand × (1 + a), temperature + b and
   wind × (1 + c), where a, b, c are slowly varying AR(1) processes (typical size ±1.5 %, ±0.6 °C and ±10 %, wind
   factor clipped to 0.7–1.3); all three irradiance components × one daily factor clipped to 0.85–1.15 (so DHI never
   exceeds GHI and nights stay 0).
5. **Derived columns** (local Year/Month/Hour/…, Season) come from the same `merge_region` code as the real data.
   **Stress_Index** is placed on the *real* data's percentile scale with the same formula, so real rows keep
   their values exactly; synthetic demand above the real maximum is capped at 100.

**How they are distinguished.** A `data_type` column marks every row as `real` or `synthetic`. Real rows come
first, unchanged and in their original order. The dashboard shows real data only unless *Real + synthetic
(demo)* is selected, and then labels the synthetic data explicitly.

**Limitations.** Synthetic rows are resampled real days with noise: each region's synthetic period is essentially one real year's
January–July re-played with small noise. It contains no new information, cannot represent events that did not
occur in that template (e.g. a new record heat wave), misaligns public holidays and ignores year-to-year load growth, and cover January–July only (5,000 hours), so pooled real + synthetic averages shift
toward winter/spring. They must never be cited as measured demand. Because each region replays one real year, real extreme
events in that year are replayed too — e.g. if Texas's template is 2021, its synthetic February repeats Winter Storm Uri.

**Validation (next cell).** Real rows are compared with the real master column by column; counts, duplicates, overlap,
missing values, physical limits, one-hour spikes and the largest hour-to-hour changes (allowed up to 5 % above the
real maximum, since noise is added on top of copied real steps) are checked, and the synthetic distributions are
compared with real data from the same calendar months.

In [27]:
# ── Synthetic demonstration rows (NOT measured data) ──────────────────────────
import numpy as np

SYN_SEED = 42
SYN_HOURS_PER_REGION = 5_000          # 4 regions × 5,000 = 20,000 synthetic rows
WEATHER_COLS = ["GHI", "DNI", "DHI", "Temperature", "Wind Speed"]
rng = np.random.default_rng(SYN_SEED)


def spike_flags(g):
    """Isolated one-hour demand spikes: > 10 % above or below BOTH neighbouring hours (g sorted by time)."""
    d, t = g["Demand_MW"], g["Timestamp"]
    contiguous = (t.diff() == pd.Timedelta("1h")) & (t.diff(-1) == -pd.Timedelta("1h"))
    up = (d > d.shift() * 1.10) & (d > d.shift(-1) * 1.10)
    down = (d < d.shift() * 0.90) & (d < d.shift(-1) * 0.90)
    return contiguous & (up | down)


def ar1(n, phi, sd):
    """Smooth AR(1) noise with stationary standard deviation `sd` (starts in its stationary state)."""
    e = rng.normal(0, sd * np.sqrt(1 - phi ** 2), n)
    x = np.empty(n); x[0] = rng.normal(0, sd)
    for i in range(1, n):
        x[i] = phi * x[i - 1] + e[i]
    return x


def real_scale_stress(values, real_demand):
    """Stress_Index of new values on the region's REAL percentile scale (same formula; ties = average rank)."""
    s = np.sort(real_demand.to_numpy()); n = len(s)
    lo, hi = np.searchsorted(s, values, "left"), np.searchsorted(s, values, "right")
    return np.clip((lo + hi - 1) / 2 / (n - 1) * 100, 0, 100)


syn_frames, syn_log = [], []
for region, cfg in REGIONS.items():
    tz = cfg["tz"]
    real_r = master[master["Region"] == region].sort_values("Timestamp").reset_index(drop=True)
    real_r["local_date"] = real_r["Timestamp_Local"].str[:10]
    counts = real_r.groupby("local_date").size()
    day_len = {d: int((pd.Timestamp(d).tz_localize(tz) + pd.DateOffset(days=1) - pd.Timestamp(d).tz_localize(tz))
                      / pd.Timedelta("1h")) for d in counts.index}
    bad_days = {d for d, n in counts.items() if n != day_len[d]} | set(real_r.loc[spike_flags(real_r), "local_date"])

    # Synthetic timeline: 5,000 consecutive hours right after the real period
    start = pd.Timestamp(f"{cfg['years'][-1] + 1}-01-01").tz_localize(tz)
    ts = pd.date_range(start.tz_convert("UTC"), periods=SYN_HOURS_PER_REGION, freq="h")
    local = ts.tz_convert(tz)
    syn = pd.DataFrame({"Timestamp": ts, "local_date": local.strftime("%Y-%m-%d"), "Hour": local.hour})
    syn_days = pd.Index(syn["local_date"].unique())

    # Template: one continuous real stretch from a seeded real year, weekday-aligned, free of bad days
    for template_year in rng.permutation(cfg["years"]):
        t0 = pd.Timestamp(f"{template_year}-01-01")
        t0 += pd.Timedelta(days=(start.dayofweek - t0.dayofweek) % 7)
        template_days = [(t0 + pd.Timedelta(days=i)).strftime("%Y-%m-%d") for i in range(len(syn_days))]
        if not set(template_days) & bad_days and set(template_days) <= set(counts.index):
            break
    else:
        raise RuntimeError(f"{region}: no clean template stretch")
    syn["template_date"] = syn["local_date"].map(dict(zip(syn_days, template_days)))

    first = real_r.drop_duplicates(["local_date", "Hour"])   # repeated fall-back hour: first copy
    have = set(zip(first["local_date"], first["Hour"]))      # DST: use the neighbouring hour if absent
    syn["template_hour"] = [h if (d, h) in have else (h - 1 if h > 0 else h + 1)
                            for d, h in zip(syn["template_date"], syn["Hour"])]
    syn = syn.merge(first[["local_date", "Hour", "Demand_MW"] + WEATHER_COLS]
                    .rename(columns={"local_date": "template_date", "Hour": "template_hour"}),
                    on=["template_date", "template_hour"], how="left", validate="many_to_one")
    assert syn[["Demand_MW"] + WEATHER_COLS].notna().all().all(), f"{region}: template lookup failed"

    # Smooth, bounded perturbations
    n = len(syn)
    solar_day = pd.Series(np.clip(1 + rng.normal(0, 0.05, len(syn_days)), 0.85, 1.15), index=syn_days)
    syn["Demand_MW"] = (syn["Demand_MW"] * (1 + ar1(n, 0.98, 0.015))).round()
    for c in ["GHI", "DNI", "DHI"]:
        syn[c] = (syn[c] * syn["local_date"].map(solar_day)).round(1)
    syn["Temperature"] = (syn["Temperature"] + ar1(n, 0.97, 0.6)).round(2)
    syn["Wind Speed"] = (syn["Wind Speed"] * np.clip(1 + ar1(n, 0.95, 0.10), 0.7, 1.3)).round(2)

    # Same feature code as the real data
    m = merge_region(syn[["Timestamp", "Demand_MW"]].assign(Region=region)[["Region", "Timestamp", "Demand_MW"]],
                     syn[["Timestamp"] + WEATHER_COLS].assign(Region=region), region, tz)
    m["Stress_Index"] = real_scale_stress(m["Demand_MW"], real_r["Demand_MW"])
    syn_frames.append(m[master.columns])
    syn_log.append({"Region": region, "Synthetic rows": len(m),
                    "Synthetic period (local)": f"{m['Timestamp_Local'].iloc[0][:16]} → {m['Timestamp_Local'].iloc[-1][:16]}",
                    "Template stretch (real, local dates)": f"{template_days[0]} → {template_days[-1]}",
                    "Excluded real days (incomplete or spike)": len(bad_days)})

synthetic = pd.concat(syn_frames, ignore_index=True)
master_out = pd.concat([master.assign(data_type="real"), synthetic.assign(data_type="synthetic")],
                       ignore_index=True)
print(pd.DataFrame(syn_log).to_string(index=False))
print(f"\nReal rows: {len(master):,}   Synthetic rows: {len(synthetic):,}   Total: {len(master_out):,}")

  Virginia: 5,000 merged rows; 0 demand hours had no matching weather hour


  Ohio: 5,000 merged rows; 0 demand hours had no matching weather hour


  Texas: 5,000 merged rows; 0 demand hours had no matching weather hour


  Georgia: 5,000 merged rows; 0 demand hours had no matching weather hour
  Region  Synthetic rows            Synthetic period (local) Template stretch (real, local dates)  Excluded real days (incomplete or spike)
Virginia            5000 2018-01-01 00:00 → 2018-07-28 08:00              2017-01-02 → 2017-07-29                                         0
    Ohio            5000 2018-01-01 00:00 → 2018-07-28 08:00              2015-01-05 → 2015-08-01                                         1
   Texas            5000 2022-01-01 00:00 → 2022-07-28 08:00              2021-01-02 → 2021-07-29                                         1
 Georgia            5000 2022-01-01 00:00 → 2022-07-28 08:00              2019-01-05 → 2019-08-01                                         4

Real rows: 105,190   Synthetic rows: 20,000   Total: 125,190


In [28]:
# ── Validation of the synthetic rows ──────────────────────────────────────────
real_part = master_out[master_out["data_type"] == "real"].drop(columns="data_type").reset_index(drop=True)
syn_part = master_out[master_out["data_type"] == "synthetic"]
def max_step(df, region, col):
    """Largest hour-to-hour change (°C for temperature, relative for demand), ignoring changes into or out of
    a suspected one-hour spike so the real reference is not inflated by source errors."""
    g = df[df["Region"] == region].sort_values("Timestamp")
    step = g[col].pct_change() if col == "Demand_MW" else g[col].diff()
    f = spike_flags(g)
    contiguous = g["Timestamp"].diff() == pd.Timedelta("1h")
    return step[contiguous & ~(f | f.shift(1, fill_value=False))].abs().max()


checks = {
    "Real rows identical to the real master (all 16 columns, same order)": real_part.equals(master.reset_index(drop=True)),
    "Synthetic row count == 4 × SYN_HOURS_PER_REGION": len(syn_part) == SYN_HOURS_PER_REGION * len(REGIONS),
    "Columns = 16 original + data_type": list(master_out.columns) == list(master.columns) + ["data_type"],
    "Dtypes of synthetic rows match real rows": (synthetic.dtypes == master.dtypes).all(),
    "No duplicate (Region, Timestamp)": not master_out.duplicated(["Region", "Timestamp"]).any(),
    "Synthetic hours start after each region's last real hour":
        all(syn_part.loc[syn_part.Region == r, "Timestamp"].min() > master.loc[master.Region == r, "Timestamp"].max()
            for r in REGIONS),
    "No missing values": not master_out.isna().any().any(),
    "Demand > 0": (syn_part["Demand_MW"] > 0).all(),
    "Irradiance >= 0 and <= 1,361 W/m²": syn_part[["GHI", "DNI", "DHI"]].ge(0).all().all()
                                          and syn_part[["GHI", "DNI"]].le(1361).all().all(),
    "DHI <= GHI (+1 W/m² rounding)": (syn_part["DHI"] <= syn_part["GHI"] + 1).all(),
    "Temperature within -60…60 °C, wind >= 0": syn_part["Temperature"].between(-60, 60).all() and (syn_part["Wind Speed"] >= 0).all(),
    "Stress_Index within 0–100": syn_part["Stress_Index"].between(0, 100).all(),
    "Per region: largest hour-to-hour temperature change <= real data's × 1.05":
        all(max_step(syn_part, r, "Temperature") <= 1.05 * max_step(master, r, "Temperature") for r in REGIONS),
    "Per region: largest hour-to-hour demand change (%) <= real data's × 1.05 (real excl. suspected spikes)":
        all(max_step(syn_part, r, "Demand_MW") <= 1.05 * max_step(master, r, "Demand_MW") for r in REGIONS),
    "No isolated one-hour spikes in synthetic demand":
        sum(spike_flags(g.sort_values("Timestamp")).sum() for _, g in syn_part.groupby("Region")) == 0,
}
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values()), "Synthetic data validation failed"

# Largest hour-to-hour changes (5 % tolerance allows for the added noise on top of copied real steps)
print()
print(pd.DataFrame([{"Region": r,
                     "Max demand step real (excl. spikes)": max_step(master, r, "Demand_MW"),
                     "Max demand step synthetic": max_step(syn_part, r, "Demand_MW"),
                     "Max temp step real (°C)": max_step(master, r, "Temperature"),
                     "Max temp step synthetic (°C)": max_step(syn_part, r, "Temperature")} for r in REGIONS]
                   ).round(4).to_string(index=False))

# Distribution check against REAL data from the same calendar months (synthetic covers Jan–Jul only)
rows = []
for r in REGIONS:
    s = syn_part[syn_part.Region == r]
    re_ = master[(master.Region == r) & master.Month.isin(s.Month.unique())]
    prof_r, prof_s = re_.groupby("Hour").Demand_MW.mean(), s.groupby("Hour").Demand_MW.mean()
    mon_r, mon_s = re_.groupby("Month").Demand_MW.mean(), s.groupby("Month").Demand_MW.mean()
    rows.append({"Region": r, "Real mean (MW)": re_.Demand_MW.mean(), "Synthetic mean (MW)": s.Demand_MW.mean(),
                 "Real std": re_.Demand_MW.std(), "Synthetic std": s.Demand_MW.std(),
                 "Real max": re_.Demand_MW.max(), "Synthetic max": s.Demand_MW.max(),
                 "Hourly-profile r": prof_r.corr(prof_s), "Monthly-mean r": mon_r.corr(mon_s),
                 "Temp–demand r real": re_.Temperature.corr(re_.Demand_MW),
                 "Temp–demand r synthetic": s.Temperature.corr(s.Demand_MW)})
dist = pd.DataFrame(rows)
print()
print(dist.round(3).to_string(index=False))

# Visual check: hourly demand profile, real (same months) vs synthetic
prof = pd.concat([
    master[master.Month.isin(syn_part.Month.unique())].assign(Data="Real (same months)"),
    syn_part.assign(Data="Synthetic (demo)"),
]).groupby(["Region", "Data", "Hour"]).Demand_MW.mean().reset_index()
fig = px.line(prof, x="Hour", y="Demand_MW", color="Data", facet_col="Region", facet_col_wrap=2,
              height=500, title="Validation: Hourly Demand Profile, Real vs Synthetic (Jan–Jul)",
              labels={"Demand_MW": "Avg Demand (MW)", "Hour": "Hour of Day (local)"},
              color_discrete_map={"Real (same months)": "steelblue", "Synthetic (demo)": "darkorange"})
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(dtick=3)
takeaway = ("Takeaway: hourly-profile correlation real vs synthetic — "
            + ", ".join(f"{r} {v:.3f}" for r, v in zip(dist.Region, dist["Hourly-profile r"]))
            + ". Synthetic rows are resampled real days; they are not measurements.")
fig.add_annotation(text=takeaway, xref="paper", yref="paper", x=0, y=-0.14, showarrow=False,
                   font=dict(size=10, color="gray"))
fig.show()

  PASS  Real rows identical to the real master (all 16 columns, same order)
  PASS  Synthetic row count == 4 × SYN_HOURS_PER_REGION
  PASS  Columns = 16 original + data_type
  PASS  Dtypes of synthetic rows match real rows
  PASS  No duplicate (Region, Timestamp)
  PASS  Synthetic hours start after each region's last real hour
  PASS  No missing values
  PASS  Demand > 0
  PASS  Irradiance >= 0 and <= 1,361 W/m²
  PASS  DHI <= GHI (+1 W/m² rounding)
  PASS  Temperature within -60…60 °C, wind >= 0
  PASS  Stress_Index within 0–100
  PASS  Per region: largest hour-to-hour temperature change <= real data's × 1.05
  PASS  Per region: largest hour-to-hour demand change (%) <= real data's × 1.05 (real excl. suspected spikes)
  PASS  No isolated one-hour spikes in synthetic demand

  Region  Max demand step real (excl. spikes)  Max demand step synthetic  Max temp step real (°C)  Max temp step synthetic (°C)
Virginia                                 0.18                       0.17              


  Region  Real mean (MW)  Synthetic mean (MW)  Real std  Synthetic std  Real max  Synthetic max  Hourly-profile r  Monthly-mean r  Temp–demand r real  Temp–demand r synthetic
Virginia       11,318.84            11,060.63  2,604.09       2,429.66 21,651.00      19,816.00              1.00            0.86                0.04                     0.25
    Ohio       14,905.43            15,575.43  2,582.34       2,761.39 24,739.00      25,080.00              1.00            0.96               -0.18                    -0.35
   Texas       43,142.71            43,746.05  9,127.92       9,103.94 73,965.00      73,136.00              1.00            0.98                0.54                     0.42
 Georgia       26,302.85            26,962.22  5,406.74       5,603.55 45,072.00      45,656.00              1.00            0.92                0.41                     0.46


## 8. Save Output Files

Four clean CSVs are written to `data/clean/`:

| File | Contents |
|------|----------|
| `master_timeseries.csv` | Hourly time series, all regions, all features, plus `data_type` (`real` / `synthetic`). Real rows come first and are unchanged |
| `im3_clean.csv` | IM3 filtered to 4 states with Region column |
| `im3_atlas_full.csv` | IM3, all states with coordinates; non-focus states labeled `Other` |
| `geo_summary.csv` | One row per region: DC counts, floor area (sq ft) by type, demand stats |

In [29]:
# Convert tz-aware Timestamps to ISO strings for CSV portability
save_master = master_out.copy()          # real rows + labelled synthetic rows
save_master["Timestamp"]       = save_master["Timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
save_master["Timestamp_Local"] = save_master["Timestamp_Local"].astype(str)

master_path = CLEAN / "master_timeseries.csv"
im3_path    = CLEAN / "im3_clean.csv"
im3_full_path = CLEAN / "im3_atlas_full.csv"
geo_path    = CLEAN / "geo_summary.csv"

save_master.to_csv(master_path, index=False)
im3_clean.to_csv(im3_path, index=False)
im3_full.to_csv(im3_full_path, index=False)
geo_summary.to_csv(geo_path, index=False)

print(f"Saved: {master_path}  ({len(save_master):,} rows: {(save_master.data_type == 'real').sum():,} real + {(save_master.data_type == 'synthetic').sum():,} synthetic)")
print(f"Saved: {im3_path}  ({len(im3_clean):,} rows)")
print(f"Saved: {im3_full_path}  ({len(im3_full):,} rows)")
print(f"Saved: {geo_path}  ({len(geo_summary):,} rows)")

# Verify saved files load correctly
check = pd.read_csv(master_path)
assert len(check) == len(master_out), "Row count mismatch after save!"
assert (check["data_type"] == "real").sum() == len(master), "Real row count changed!"
print("Save verification: OK")

Saved: data/clean/master_timeseries.csv  (125,190 rows: 105,190 real + 20,000 synthetic)
Saved: data/clean/im3_clean.csv  (577 rows)
Saved: data/clean/im3_atlas_full.csv  (1,479 rows)
Saved: data/clean/geo_summary.csv  (4 rows)
Save verification: OK


## Final Summary

In [30]:
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"Real rows (used for all analysis above): {len(master):,}")
print(f"Synthetic demonstration rows (data_type == 'synthetic', not measured): {len(synthetic):,}")
print(f"Rows in master_timeseries.csv: {len(master_out):,}")
print()
region_summary = master.groupby("Region").agg(
    rows=("Demand_MW","count"),
    years=("Year", lambda x: f"{x.min()}-{x.max()}"),
    demand_avg=("Demand_MW","mean"),
    demand_peak=("Demand_MW","max"),
    stress_p95=("Stress_Index", lambda x: x.quantile(0.95)),
).round(1).reset_index()
print(region_summary.to_string(index=False))
print()
print("Data gaps / assumptions:")
print("  - NSRDB provides one lat/lon point per region (single weather station proxy).")
print("  - IM3 point records have no sqft; campus+building sqft reported separately to avoid double-counting.")
print("  - Virginia & Ohio demand is 2015-2017; Texas & Georgia is 2019-2021 (different calendar years).")
print("  - Stress Index is region-relative; cross-region comparisons are not meaningful.")
print("  - All timestamps are UTC interval starts; PJM and EIA-930 hour-ending labels shifted by 1 h.")
print("  - Missing demand hours are dropped and reported above, never filled.")
print("  - IM3 is a Feb 2026 snapshot; demand covers 2015-2021.")
print("  - No prediction is performed; all results are descriptive.")
print("  - Synthetic rows are for dashboard demonstration only and are excluded from every analysis in this notebook.")
print()
print("Run commands:")
print("  Notebook:   .venv/bin/jupyter nbconvert --to notebook --execute dav_project.ipynb")
print("  Dashboard:  .venv/bin/streamlit run app.py")

FINAL SUMMARY
Real rows (used for all analysis above): 105,190
Synthetic demonstration rows (data_type == 'synthetic', not measured): 20,000
Rows in master_timeseries.csv: 125,190

  Region  rows     years  demand_avg  demand_peak  stress_p95
 Georgia 26279 2019-2021   26,574.20    45,515.00       95.00
    Ohio 26304 2015-2017   14,712.40    24,739.00       95.00
   Texas 26303 2019-2021   43,994.00    74,533.00       95.00
Virginia 26304 2015-2017   11,117.10    21,651.00       95.00

Data gaps / assumptions:
  - NSRDB provides one lat/lon point per region (single weather station proxy).
  - IM3 point records have no sqft; campus+building sqft reported separately to avoid double-counting.
  - Virginia & Ohio demand is 2015-2017; Texas & Georgia is 2019-2021 (different calendar years).
  - Stress Index is region-relative; cross-region comparisons are not meaningful.
  - All timestamps are UTC interval starts; PJM and EIA-930 hour-ending labels shifted by 1 h.
  - Missing demand hours 